[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/navjotts/ML-experiments/blob/master/10%20-%20High%20Dimensionality%20and%20Manual%20Clustering/High_Dimensionality_and_Manual_Clustering.ipynb)

# Purpose
The purpose of this experiment is to showcase

1.   how High Dimensionality makes it harder to pull out any meaning from data ([Curse of dimensionality](https://en.wikipedia.org/wiki/Curse_of_dimensionality)) 
2.   a manual approach to Unsupervised Clustering


We dabble with the Titanic data, and try to find some clusters **manually** - in an attempt to find any meaningful information about the main prediction which concerns the Titanic tragedy - `died v/s survived`.


---



### Data

In [0]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

titanic = sns.load_dataset('titanic')
titanic = titanic.drop(['alive','adult_male','who','class','embark_town'], axis=1)
titanic['embarked'] = titanic['embarked'].fillna(method='ffill')
titanic = titanic.drop(['deck'], axis=1)
titanic['age'] = titanic['age'].fillna(method='ffill')
for label in ['embarked', 'sex', 'alone']:
    titanic[label] = LabelEncoder().fit_transform(titanic[label])
    
titanic.head()

# Part 1

### As a First step, let's

1. Calculate the centroid of the survivors
2. Calculate the centroid of the casualties
3. Calculate the average distance between each survivor
4. Calculate the average distance between each casualty
5. Calculate the distance between the two centroids


In [0]:
# 1st separate out survivors and casualities datasets
survivors = titanic[titanic['survived']==1]
casualities = titanic[titanic['survived']==0]

survivors = survivors.drop(['survived'], axis=1)
casualities = casualities.drop(['survived'], axis=1)

features = titanic.drop(['survived'], axis=1)

In [0]:
# centroid of a given data set in a given axes
def centroid(data, axes):
  centroid = []  
  for axis in axes:
    points = data[axis]
    centroid.append(points.mean())  
  return centroid

In [0]:
# centroid of the survivors
survivors_centroid = centroid(survivors, features.columns.values)
print(survivors_centroid)

In [0]:
# centroid of the casualities
casualities_centroid = centroid(casualities, features.columns.values)
print(casualities_centroid)

In [0]:
# distance b/w 2 points
def point_distance(pointA, pointB):
  return np.linalg.norm(np.array(pointA)-np.array(pointB))

In [0]:
# distance b/w centroids
centroids_distance = point_distance(survivors_centroid, casualities_centroid)
print(centroids_distance)

In [0]:
# average distance between 2 points in a data set
def average_dist(data, axes):
  data = pd.DataFrame(data, columns=axes)  
  pairs = []
  for i in range(len(data)):
    for j in range(i+1, len(data)):
      pairs.append([data.iloc[i].values, data.iloc[j].values])
  
  distances = [point_distance(pointA, pointB) for (pointA, pointB) in pairs]    
  return np.array(distances).mean()

In [0]:
# average distance between each survivor
print(average_dist(survivors, features.columns.values))

In [0]:
# average distance between each casualty
print(average_dist(casualities, features.columns.values))

### Results:

*   distance b/w centroids: **26.365200774120844**
*   avg distance b/w each survivor: **61.79494312367423**
*   avg distance b/w each casualty: **32.08065292422003**   
*   what the above potentially means is that the **2 clusters are overlapping** as the distance b/w any 2 survivors is much more than the distance b/w the 2 centroids


# Next
What we want to do is to find the set of dimensions where: 

**the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance.**

The set of dimensions that maximizes centroid distance and minimizes the cluster distances gives us a good shot at making clear clusters from which we could make some useful predictions.

# Part 2: Automation

### Step 1

There can be a lot of sets of dimensions, and for each of those sets, we are concerned with 3 things:

*   centroids_distance
*   avg_survivor_distance
*   avg_casualty_distance

Generate a new DataFrame (with these 3 as columns, and each unique set_of_dimension as a key), so that we can perform experiements on this dataset without having to do the distance calculations repeatedly.


In [0]:
import itertools

MAX_DIMENSIONS = len(features.columns)
MIN_DIMENSIONS = 1

def centroid_and_cluster_distances(num_of_dim):
  data = pd.DataFrame([])
  
  possible_combinations = list(itertools.combinations(np.arange(0, MAX_DIMENSIONS), num_of_dim))
  print("possible col_number combinations in " + str(num_of_dim) + " dimensions: " + str(possible_combinations))
  for each in possible_combinations:
    feature_cols = [features.columns.values[i] for i in each]
    print("========== trying feature columns: " + str(feature_cols))
    centroids_distance = point_distance(centroid(survivors, feature_cols), centroid(casualities, feature_cols))
    survivor_distance = average_dist(survivors, feature_cols)
    casualty_distance = average_dist(casualities, feature_cols)
    print("========== " + str(centroids_distance) + ", " + str(survivor_distance) + ", " + str(casualty_distance))
    data_entry = pd.DataFrame([[num_of_dim, centroids_distance, survivor_distance, casualty_distance]],
                              columns=['Dimension Size', 'Centroids Distance', 'Avg Survivor Distance', 'Avg Casualty Distance'],
                              index=[str(feature_cols)])
    data_entry.index.rename('Dimension Set', inplace=True)    
    data = data.append(data_entry)
  
  return data

In [0]:
distances = pd.DataFrame([])

for dim in range(MIN_DIMENSIONS, MAX_DIMENSIONS+1):
  distances = distances.append(centroid_and_cluster_distances(dim))

In [0]:
distances.iloc[np.r_[0:5, -5:0]]

In [0]:
from google.colab import files

def save_data_to_local(data, file):  
  with open(file, 'w') as f:
    f.write('')  
  data.to_csv(file)
  files.download(file)

In [0]:
filename = 'distances-High_Dimensional_Research.csv'

In [0]:
save_data_to_local(distances, filename)

### Load dataset from local if required (we don't want to calculate distances again and again)

In [0]:
from google.colab import files
uploaded = files.upload()
  
import io
distances = pd.read_csv(io.StringIO(uploaded[filename].decode('utf-8')))
distances.head()

### Step 2
Once we have our dataset of concerned points, we need to find _'the set of dimensions where: the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance'_

In [0]:
optimal = distances[(distances['Avg Survivor Distance']<distances['Centroids Distance']) & (distances['Avg Casualty Distance']<distances['Centroids Distance'])]
optimal

### Result
**['sex']** is the only set of dimensions, where this holds true: _'the mean distance between each survivor and the mean distance between each casualty is less than the centroid distance'_

## Part 2: "Best" sets of dimension
which "maximize the centroid distance and minimize the cluster distances"

### Thought process

*   ideally avg_survivor_distance/centroids_distance<1 && avg_casuality_distance/centroids_distance<1
*   what we want is to minimize both avg_survivor_distance/centroids_distance & avg_casuality_distance/centroids_distance

### Steps
*   get 2 different datasets of 20 options having smallest values of avg_survivor_distance/centroids_distance & avg_casuality_distance/centroids_distance respectively
*   and see if there are any common rows in the 2 datasets




### Implementation

In [0]:
# add 2 more columns to our dataset
distances['Survivor/Centroid ratio'] = distances['Avg Survivor Distance']/distances['Centroids Distance']
distances['Casualty/Centroid ratio'] = distances['Avg Casualty Distance']/distances['Centroids Distance']

In [0]:
distances.iloc[np.r_[0:5, -5:0]]

In [0]:
survivor_by_centroid_min = distances.sort_values(by=['Survivor/Centroid ratio'])
survivor_by_centroid_min.head(20)

In [0]:
casualty_by_centroid_min = distances.sort_values(by=['Casualty/Centroid ratio'])
casualty_by_centroid_min.head(20)

In [0]:
survivor_set = survivor_by_centroid_min.iloc[0:20, 0]
casualty_set = casualty_by_centroid_min.iloc[0:20, 0]

In [0]:
# find the intersection of the 2 sets
idx1 = survivor_set.index
idx2 = casualty_set.index
common_indexes = idx1.intersection(idx2)
print(common_indexes.values)

In [0]:
print("'Best' sets of dimensions:")
for ind in common_indexes.values:
  print(distances.iloc[ind, 0])